# 67 · Chẩn đoán sau `R2a` + quét ensemble trên val

**0 GPU. Vài phút. Tự đủ — không cần notebook nào chạy trước.**

Gom mọi phép đo rời rạc sau khi `64`/`64b` xong và chạm test lần 2.

```
TEST hien tai   0.2625   (duong ong 0.2583 + cnn, w tier 0.40/0.40/0.00)
vo dich (doi A) 0.281    cach 0.0185
```

## Vì sao được phép quét trên val

```
truoc:  chon tren LAT CHON  ->  doc tren VAL
gio:    chon tren VAL       ->  doc tren TEST
```

Val chưa bao giờ nằm trong train ⇒ độc lập với test. Việc val đã bị dùng thích nghi lâu chỉ hỏng
nó với tư cách **thước đo**, không hỏng với tư cách **tập chọn**. Thứ duy nhất còn cấm là **chọn
TRÊN test**.

> **Notebook này KHÔNG chạm test.** Nó chốt cấu hình trên val; test đọc một lượt riêng, sau.

## Nội dung

```
C1  kiem TUAN THU nhan      5 nhan zero-shot Task 3 · nguon goc nhan dev/test
C2  TIER CHINH THUC         >10% / 1-10% / <1% tren train, phan ra khoang cach theo tier
C3  CHAN DOAN TUNG NHAN     ghep do chinh xac nhan (Table 8 cua paper) -> 3 nhom thuoc
C4  QUET tren val           hang vs xac suat · w theo tier · trong so epoch · cnn_ms
                            + 3 co che chong overfit val
C5  TONG HOP                cau hinh de xuat cho lan doc test ke tiep
```

**Nguồn tham chiếu:** `paper/2506.07984v1.pdf` (Lin et al., arXiv:2506.07984) — Table 1 (split),
Table 4–5 (bảng xếp hạng + tier), Table 8 (độ chính xác nhãn tự động).


## C0 · Nạp dữ liệu và dự đoán (0 GPU)

In [1]:
import os, json, glob, re, itertools
import numpy as np, pandas as pd

try:
    from google.colab import drive
    drive.mount('/content/drive'); DATA_DIR = '/content/drive/MyDrive/MIMIC_CXR/planB_out'
except Exception:
    DATA_DIR = './planB_out'
OUT_DIR = os.path.join(DATA_DIR, 'cxrlt_2024')
EMB_DIR = os.path.join(OUT_DIR, 'embeddings')

INV   = json.load(open(os.path.join(OUT_DIR, 'cxrlt_inventory.json')))
LAB40 = INV['labels_task1']; C = len(LAB40)
TIER_TA = {t: np.array([LAB40.index(l) for l in INV['tier'][t]]) for t in ('head','medium','tail')}

ORDER = pd.read_parquet(os.path.join(EMB_DIR, 'emb_index.parquet'))
Y = (pd.read_parquet(os.path.join(OUT_DIR, 'cxrlt_index.parquet'), columns=['dicom_id'] + LAB40)
       .set_index('dicom_id').loc[ORDER.dicom_id].reset_index())
assert (Y.dicom_id.to_numpy() == ORDER.dicom_id.to_numpy()).all()
YM  = Y[LAB40].to_numpy('float32')
SUB = ORDER.subject_id.to_numpy(); SPLIT = ORDER.split.to_numpy()
TR, VA = SPLIT == 'train', SPLIT == 'val'; TE = ~(TR | VA)
print(f'{len(ORDER):,} anh | train {TR.sum():,} | val {VA.sum():,} | test {TE.sum():,}')
print('paper Table 1 : train 258,871 | dev 39,293 | test 78,946')

from sklearn.metrics import average_precision_score
def ap_cols(Yc, Sc):
    o = np.full(Yc.shape[1], np.nan)
    for j in range(Yc.shape[1]):
        y, s = Yc[:, j], Sc[:, j]; n1 = y.sum()
        if n1 == 0 or n1 == len(y): continue
        r = np.argsort(-s, kind='stable'); yy, ss = y[r], s[r]
        tp, fp = np.cumsum(yy), np.cumsum(1 - yy)
        last = np.r_[np.diff(ss) != 0, True]; tp, fp = tp[last], fp[last]
        pr, rc = tp / (tp + fp), tp / n1
        o[j] = float((np.diff(np.r_[0.0, rc]) * pr).sum())
    return o
_r6 = np.random.default_rng(6); _y = (_r6.random((3000, C)) < 0.2).astype('float32')
_s = _r6.random((3000, C)).astype('float32')
assert np.nanmax(np.abs(ap_cols(_y, _s) - np.array(
    [average_precision_score(_y[:, j], _s[:, j]) for j in range(C)]))) < 1e-6
print('cong: ap_cols khop sklearn')

# ---- nap DU DOAN, giu RIENG tung file de C4 can lai theo epoch ----
def _mau(goc, sub):
    base = os.path.dirname(os.path.join(OUT_DIR, goc)); nm = os.path.basename(goc)
    return [os.path.join(base, d, sub) for d in os.listdir(base)
            if (d == nm or d.startswith(nm + ' (')) and os.path.isdir(os.path.join(base, d, sub))]

def _tap(goc, run, arm, pre, n_can):
    '''Tra ve dict (seed, epoch) -> mang du doan. Khu trung ten giua cac thu muc sao chep.'''
    uq = {}
    for d in _mau(goc, run):
        for f in sorted(glob.glob(os.path.join(d, f'{pre}_{arm}_s*_ep*.npy'))):
            uq.setdefault(os.path.basename(f), f)
    out = {}
    for bn, f in sorted(uq.items()):
        mm = re.search(r'_s(\d+)_ep(\d+)\.npy$', bn)
        P = np.load(f).astype('float32')
        assert P.shape == (n_can, C), f'{bn}: {P.shape} != {(n_can, C)}'
        out[(int(mm.group(1)), int(mm.group(2)))] = P
    assert out, f'khong thay {pre}_{arm}_* trong {goc}/{run}'
    return out

n_va = int(VA.sum())
NGUON = {'pipe':     ('stage4_nb64',  'r2a', 'tpb'),
         'cnn_full': ('stage4_nb64b', 'r2a', 'cnn_full'),
         'cnn_ms':   ('stage4_nb41f', 'r2',  'cnn_ms')}     # thanh vien thu 3, co the chua co
FILE = {}
for k, (g, run, arm) in NGUON.items():
    try:
        FILE[k] = _tap(g, run, arm, 'p', n_va)
        _eps = sorted({e for _, e in FILE[k]}); _sd = sorted({s for s, _ in FILE[k]})
        print(f'  {k:<10} {len(FILE[k]):>3} file | seed {_sd} | epoch {_eps[0]}-{_eps[-1]}')
    except Exception as e:
        print(f'  {k:<10} KHONG NAP DUOC ({type(e).__name__}) -> bo qua o C4')
ENS = {k: np.mean(list(v.values()), 0) for k, v in FILE.items()}
Y_VA = YM[VA]
print('\nensemble deu (moi epoch x moi seed):')
for k, P in ENS.items(): print(f'  {k:<10} macro {float(np.nanmean(ap_cols(Y_VA, P))):.4f}')

Mounted at /content/drive
377,110 anh | train 258,871 | val 39,293 | test 78,946
paper Table 1 : train 258,871 | dev 39,293 | test 78,946
cong: ap_cols khop sklearn
  pipe        20 file | seed [42, 43] | epoch 1-10
  cnn_full    16 file | seed [42, 43] | epoch 1-8
  cnn_ms      20 file | seed [42, 43] | epoch 1-10

ensemble deu (moi epoch x moi seed):
  pipe       macro 0.2663
  cnn_full   macro 0.2463
  cnn_ms     macro 0.2419


## C1 · Kiểm tuân thủ nhãn

Paper §2.2.1: 5 phát hiện **giữ lại cho Task 3 zero-shot**, không có trong tập huấn luyện Task 1.
Và §2.2: *"participants were provided with labels **only for the training set**"* — ta chấm cả
dev lẫn test nên phải biết nhãn từ đâu.

In [2]:
ZS    = ['Bulla', 'Cardiomyopathy', 'Hilum', 'Osteopenia', 'Scoliosis']
MOI14 = ['Adenopathy', 'Azygos Lobe', 'Clavicle Fracture', 'Fissure', 'Hydropneumothorax',
         'Infarction', 'Kyphosis', 'Lobar Atelectasis', 'Pleural Other', 'Pulmonary Embolism',
         'Pulmonary Hypertension', 'Rib Fracture', 'Round Atelectasis', 'Tuberculosis']
_ch = lambda s: re.sub(r'[^a-z]', '', s.lower())      # Round(ed) -> rounded
_ta = {_ch(l): l for l in LAB40}
_tim = lambda s: [v for k, v in _ta.items() if k == _ch(s) or _ch(s) in k or k in _ch(s)]

print('=== 5 nhan ZERO-SHOT (Task 3) — co pham khong ===')
_pham = [(z, _tim(z)) for z in ZS if _tim(z)]
for z in ZS: print(f'  {z:<16} -> ' + (f'!! CO: {_tim(z)}' if _tim(z) else 'khong co  OK'))
assert not _pham, f'PHAM nhan zero-shot: {_pham}'
print(f'\n  KHONG pham. LAB40 = 45 - 5 = {C} nhan cua Task 1.  OK')

print('\n=== 14 nhan MOI cua 2024 co trong LAB40 khong ===')
_thieu = [m for m in MOI14 if not _tim(m)]
for m in MOI14: print(f'  {m:<24} -> ' + (_tim(m)[0] if _tim(m) else '!! KHONG THAY'))
print(f'\n  khop {14-len(_thieu)}/14' + (f' | THIEU {_thieu}' if _thieu else ''))
print('  ke toan: 40 = 14 (MIMIC goc) + 12 (CXR-LT 2023) + 14 (moi 2024)')
print('  -> dung 14 nhan moi KHONG co trong Table 8 (bang do chi phu 26 nhan cua 2023)')

print('\n=== NGUON GOC nhan dev/test ===')
_u = np.unique(YM)
print(f'  gia tri phan biet trong YM: {_u[:5]} ({len(_u)} gia tri)')
_pr = pd.DataFrame({'train': YM[TR].mean(0), 'dev': YM[VA].mean(0), 'test': YM[TE].mean(0)},
                   index=LAB40)
for c in ('dev', 'test'):
    _l = ((_pr[c] - _pr.train) / _pr.train.clip(lower=1e-9)).abs()
    print(f'  |lech ti le duong {c} vs train|  TB {_l.mean():.2%}  max {_l.max():.2%}'
          f'  ({_l.idxmax()})')
print('  chia theo BENH NHAN nen lech phai NHO. Lech nho -> nhan dev/test CUNG NGUON voi train.')
print('\n  !! Ta GIU nhan test -> day la danh gia HOI CUU, khong phai nop mu.')
print('     Thu thay the cho tinh mu la ky luat cham test (co nhat ky). PHAI neu trong bai.')

=== 5 nhan ZERO-SHOT (Task 3) — co pham khong ===
  Bulla            -> khong co  OK
  Cardiomyopathy   -> khong co  OK
  Hilum            -> khong co  OK
  Osteopenia       -> khong co  OK
  Scoliosis        -> khong co  OK

  KHONG pham. LAB40 = 45 - 5 = 40 nhan cua Task 1.  OK

=== 14 nhan MOI cua 2024 co trong LAB40 khong ===
  Adenopathy               -> Adenopathy
  Azygos Lobe              -> Azygos Lobe
  Clavicle Fracture        -> Clavicle Fracture
  Fissure                  -> Fissure
  Hydropneumothorax        -> Hydropneumothorax
  Infarction               -> Infarction
  Kyphosis                 -> Kyphosis
  Lobar Atelectasis        -> Atelectasis
  Pleural Other            -> Pleural Other
  Pulmonary Embolism       -> Pulmonary Embolism
  Pulmonary Hypertension   -> Pulmonary Hypertension
  Rib Fracture             -> Fracture
  Round Atelectasis        -> Atelectasis
  Tuberculosis             -> Tuberculosis

  khop 14/14
  ke toan: 40 = 14 (MIMIC goc) + 12 (CXR-LT 2

## C2 · Tier theo định nghĩa CHÍNH THỨC, và khoảng cách nằm ở đâu

Paper §3.3: chia theo prevalence **trên tập train** — `head >10%`, `medium 1–10%`, `tail <1%`,
ra `9 / 14 / 16` nhãn *(tổng 39 — thiếu 1, nhiều khả năng lỗi đánh máy)*.

Tier của ta (`nb28`) chia theo **số ca** nên ra `13 / 16 / 11` — **mọi so sánh tier trước đây đều
lệch thang.**

In [3]:
prev = YM[TR].mean(0)
TIER = {'head':   np.where(prev > 0.10)[0],
        'medium': np.where((prev >= 0.01) & (prev <= 0.10))[0],
        'tail':   np.where(prev < 0.01)[0]}
print(f'tier CHINH THUC : ' + str({k: len(v) for k, v in TIER.items()})
      + f'  tong {sum(len(v) for v in TIER.values())}')
print(f'paper ghi       : head 9 | medium 14 | tail 16  tong 39  <- thieu 1')
print(f'tier CU cua ta  : ' + str({k: len(v) for k, v in TIER_TA.items()}))

W_TIER = {'head': 0.40, 'medium': 0.40, 'tail': 0.00}   # KHOA tu 61, theo tier CU
def tron(Po, Pc, w, tier):
    Q = np.array(Po, copy=True)
    for t, cols in tier.items(): Q[:, cols] = (1 - w[t]) * Po[:, cols] + w[t] * Pc[:, cols]
    return Q
BV = tron(ENS['pipe'], ENS['cnn_full'], W_TIER, TIER_TA)
apV = ap_cols(Y_VA, BV)

# so tren TEST: doc lai file pte_ (KHONG phai cham moi — con so nay da cong bo o 61)
try:
    _pt = np.mean(list(_tap('stage4_nb64',  'r2a', 'tpb',      'pte', int(TE.sum())).values()), 0)
    _ct = np.mean(list(_tap('stage4_nb64b', 'r2a', 'cnn_full', 'pte', int(TE.sum())).values()), 0)
    apT = ap_cols(YM[TE], tron(_pt, _ct, W_TIER, TIER_TA)); CO_TEST = True
except Exception as e:
    print(f'khong nap duoc pte_ ({type(e).__name__}) -> chi bao cao val'); CO_TEST = False

VD = {'head': 0.567, 'medium': 0.263, 'tail': 0.136}    # doi A, Table 5
print(f'\n{"":<9}{"n":>4}{"val":>9}' + (f'{"test":>9}' if CO_TEST else '')
      + f'{"doi A":>8}' + (f'{"cach":>9}' if CO_TEST else ''))
for t in ('head', 'medium', 'tail'):
    c = TIER[t]; v = float(np.nanmean(apV[c]))
    row = f'  {t:<7}{len(c):>4}{v:>9.4f}'
    if CO_TEST:
        s = float(np.nanmean(apT[c])); row += f'{s:>9.4f}{VD[t]:>8.3f}{VD[t]-s:>+9.4f}'
    else: row += f'{VD[t]:>8.3f}'
    print(row)
_mv = float(np.nanmean(apV))
print(f'  {"macro":<7}{C:>4}{_mv:>9.4f}'
      + (f'{float(np.nanmean(apT)):>9.4f}{0.281:>8.3f}{0.281-float(np.nanmean(apT)):>+9.4f}'
         if CO_TEST else f'{0.281:>8.3f}'))

if CO_TEST:
    print('\n=== KHOANG CACH MACRO DEN DOI A, phan ra theo tier ===')
    tong = 0.0
    for t in ('head', 'medium', 'tail'):
        c = TIER[t]; g = (VD[t] - float(np.nanmean(apT[c]))) * len(c) / C; tong += g
        print(f'  {t:<7} ({len(c):>2} nhan)  {g:+.4f}  = ({VD[t]:.3f} - '
              f'{np.nanmean(apT[c]):.4f}) x {len(c)}/{C}   [{g/max(0.281-float(np.nanmean(apT)),1e-9):.0%}]')
    print(f'  {"cong":<7}            {tong:+.4f}')
    print('\n  CHUYEN GIAO val -> test (dung de biet tin val toi dau khi CHON):')
    for t in ('head', 'medium', 'tail'):
        c = TIER[t]; a, b = float(np.nanmean(apV[c])), float(np.nanmean(apT[c]))
        print(f'    {t:<7} {a:.4f} -> {b:.4f}   ti le {b/max(a,1e-9):.3f}')

tier CHINH THUC : {'head': 9, 'medium': 15, 'tail': 16}  tong 40
paper ghi       : head 9 | medium 14 | tail 16  tong 39  <- thieu 1
tier CU cua ta  : {'head': 13, 'medium': 16, 'tail': 11}

            n      val     test   doi A     cach
  head      9   0.5651   0.5663   0.567  +0.0007
  medium   15   0.2342   0.2286   0.263  +0.0344
  tail     16   0.1399   0.1234   0.136  +0.0126
  macro    40   0.2709   0.2625   0.281  +0.0185

=== KHOANG CACH MACRO DEN DOI A, phan ra theo tier ===
  head    ( 9 nhan)  +0.0002  = (0.567 - 0.5663) x 9/40   [1%]
  medium  (15 nhan)  +0.0129  = (0.263 - 0.2286) x 15/40   [70%]
  tail    (16 nhan)  +0.0050  = (0.136 - 0.1234) x 16/40   [27%]
  cong               +0.0181

  CHUYEN GIAO val -> test (dung de biet tin val toi dau khi CHON):
    head    0.5651 -> 0.5663   ti le 1.002
    medium  0.2342 -> 0.2286   ti le 0.976
    tail    0.1399 -> 0.1234   ti le 0.882


## C3 · Chẩn đoán từng nhãn — yếu vì **nhiễu nhãn** hay vì **khó**

Table 8 của paper đo độ chính xác của nhãn tự động trên tập vàng 406 ảnh (26 nhãn của CXR-LT 2023).
Nhãn có precision thấp thì **augment và kiến trúc không cứu được** — phải xử lý ở tầng nhãn.

In [4]:
PREC = {   # Table 8: (rule-based, GPT-4o) micro-precision, tap vang 406 anh
 'Atelectasis': (0.611, 0.590), 'Calcification of the Aorta': (1.000, 0.857),
 'Cardiomegaly': (0.769, 0.938), 'Consolidation': (0.816, 0.849), 'Edema': (0.638, 0.804),
 'Emphysema': (0.609, 0.639), 'Enlarged Cardiomediastinum': (0.583, 1.000),
 'Fibrosis': (0.667, 0.682), 'Fracture': (0.870, 0.937), 'Hernia': (0.633, 0.810),
 'Infiltration': (0.261, 0.889), 'Lung Lesion': (0.161, 0.000), 'Lung Opacity': (0.853, 0.984),
 'Mass': (0.513, 0.810), 'Normal': (0.917, 0.972), 'Nodule': (0.821, 0.844),
 'Pleural Effusion': (0.798, 0.812), 'Pleural Other': (0.810, 0.500),
 'Pleural Thickening': (1.000, 0.815), 'Pneumomediastinum': (0.875, 0.889),
 'Pneumonia': (0.191, 0.435), 'Pneumoperitoneum': (0.676, 0.840),
 'Pneumothorax': (0.563, 0.865), 'Subcutaneous Emphysema': (0.955, 0.889),
 'Support Devices': (0.948, 0.933), 'Tortuous Aorta': (0.958, 0.861)}
NG_BAN = 0.60      # TB toan bo bang = 0.711

_ap = apT if CO_TEST else apV
_nh = 'test' if CO_TEST else 'val'
_t = {j: t for t, c in TIER.items() for j in c}
R = [dict(lb=l, ap=float(_ap[j]), prev=float(prev[j])*100, tier=_t[j],
          prec=PREC.get(l, (None,))[0], gpt=PREC.get(l, (None, None))[1])
     for j, l in enumerate(LAB40)]

for tn in ('medium', 'tail', 'head'):
    print(f'\n=== {tn.upper()} ({len(TIER[tn])} nhan) — AP tren {_nh} ===')
    print(f'{"nhan":<28}{"AP":>8}{"prev%":>8}{"prec":>7}{"GPT4o":>7}   chan doan')
    for r in sorted([r for r in R if r['tier'] == tn], key=lambda z: z['ap']):
        cd = ('chua co so nhan' if r['prec'] is None else
              f'NHAN BAN (prec {r["prec"]:.2f}) -> augment KHONG cuu' if r['prec'] < NG_BAN else
              'nhan SACH ma AP thap -> BAI TOAN KHO' if r['ap'] < 0.15 else 'binh thuong')
        print(f'  {r["lb"]:<26}{r["ap"]:>8.4f}{r["prev"]:>8.3f}'
              + (f'{r["prec"]:>7.3f}{r["gpt"]:>7.3f}' if r['prec'] is not None
                 else f'{"-":>7}{"-":>7}') + f'   {cd}')

ban  = [r for r in R if r['prec'] is not None and r['prec'] < NG_BAN]
sach = [r for r in R if r['prec'] is not None and r['prec'] >= NG_BAN and r['ap'] < 0.15]
chua = [r for r in R if r['prec'] is None and r['ap'] < 0.15]
_du  = lambda rs: sum(max(0.30 - r['ap'], 0) for r in rs) / C
print('\n' + '=' * 74)
print(f'A. NHAN BAN (prec < {NG_BAN}): {len(ban)} nhan — gan lai nhan bang LLM, hoac ha trong so loss')
for r in sorted(ban, key=lambda z: z['prec']):
    print(f'   {r["lb"]:<26} AP {r["ap"]:.4f} | prec {r["prec"]:.3f} -> GPT4o {r["gpt"]:.3f} | {r["tier"]}')
print(f'\nB. NHAN SACH ma AP < 0.15: {len(sach)} nhan — augment / du lieu ngoai / loss co trong so')
for r in sorted(sach, key=lambda z: z['ap']):
    print(f'   {r["lb"]:<26} AP {r["ap"]:.4f} | prec {r["prec"]:.3f} | prev {r["prev"]:.3f}% | {r["tier"]}')
print(f'\nC. CHUA CO SO NHAN, AP < 0.15: {len(chua)} nhan (14 nhan moi 2024)')
for r in sorted(chua, key=lambda z: z['ap']):
    print(f'   {r["lb"]:<26} AP {r["ap"]:.4f} | prev {r["prev"]:.3f}% | {r["tier"]}')
print(f'\ntran neu MOI nhan nhom B+C len 0.30: macro +{_du(sach+chua):.4f}')
print(f'tran neu MOI nhan nhom A  len 0.30: macro +{_du(ban):.4f}')
print('  (0.30 chi de UOC DO LON, khong phai muc tieu)')


=== MEDIUM (15 nhan) — AP tren test ===
nhan                              AP   prev%   prec  GPT4o   chan doan
  Infiltration                0.0644   3.897  0.261  0.889   NHAN BAN (prec 0.26) -> augment KHONG cuu
  Tortuous Aorta              0.0742   1.289  0.958  0.861   nhan SACH ma AP thap -> BAI TOAN KHO
  Pleural Thickening          0.1248   1.264  1.000  0.815   nhan SACH ma AP thap -> BAI TOAN KHO
  Adenopathy                  0.1349   1.317      -      -   chua co so nhan
  Calcification of the Aorta  0.1492   1.637  1.000  0.857   nhan SACH ma AP thap -> BAI TOAN KHO
  Fissure                     0.1528   1.083      -      -   chua co so nhan
  Granuloma                   0.1762   1.145      -      -   chua co so nhan
  Nodule                      0.1850   2.909  0.821  0.844   binh thuong
  Rib Fracture                0.2097   3.445      -      -   chua co so nhan
  Emphysema                   0.2258   1.414  0.609  0.639   binh thuong
  Consolidation               0.2515 

## C4 · Quét trên val — bốn phương án, ba cơ chế chống overfit

Val là tập chọn hợp lệ, nhưng nó **có nhiễu**, nhất là ở tail. Ba cơ chế:

```
a) chon ROBUST     bootstrap val 200 lan (cum = benh nhan), lay gia tri THANG NHIEU LAN NHAT
b) co ngot ve prior  w~ = (n*w_val + k*0.40) / (n + k)
c) chiet khau tier   nhan voi ti le chuyen giao val->test DA DO o C2
```

In [5]:
WG   = np.round(np.arange(0, 1.001, 0.05), 2)
B_BS = 200
SUB_VA = SUB[VA]; _uq = np.unique(SUB_VA); _ix = {u: np.where(SUB_VA == u)[0] for u in _uq}
_rng = np.random.default_rng(11)
BOOT = [np.concatenate([_ix[u] for u in _rng.choice(_uq, len(_uq), replace=True)])
        for _ in range(B_BS)]
mac = lambda P, ii=None: float(np.nanmean(ap_cols(Y_VA if ii is None else Y_VA[ii],
                                                  P if ii is None else P[ii])))

def hang(P):
    '''Trung binh HANG thay cho trung binh xac suat — AP la chi so XEP HANG.'''
    from scipy.stats import rankdata
    return rankdata(P, axis=0) / P.shape[0]

print('=== 4.1 · TRUNG BINH HANG vs TRUNG BINH XAC SUAT ===')
_H = {k: np.mean([hang(v) for v in FILE[k].values()], 0) for k in FILE}
for k in FILE:
    print(f'  {k:<10} xac suat {mac(ENS[k]):.4f} | hang {mac(_H[k]):.4f}'
          f'   {mac(_H[k])-mac(ENS[k]):+.4f}')

print('\n=== 4.2 · TRONG SO THEO EPOCH (hien tai: DEU) ===')
def theo_ep(d, che):
    ks = sorted(d); eps = np.array([e for _, e in ks], 'float32')
    if   che == 'deu':    w = np.ones_like(eps)
    elif che == 'cuoi3':  w = (eps >= eps.max() - 2).astype('float32')
    elif che == 'cuoi5':  w = (eps >= eps.max() - 4).astype('float32')
    elif che == 'tuyen':  w = eps - eps.min() + 1.0
    w = w / w.sum()
    return np.tensordot(w, np.stack([d[k] for k in ks]), axes=1)
EPW = {}
for che in ('deu', 'cuoi5', 'cuoi3', 'tuyen'):
    EPW[che] = {k: theo_ep(FILE[k], che) for k in FILE}
    print(f'  {che:<7} ' + ' | '.join(f'{k} {mac(EPW[che][k]):.4f}' for k in FILE))

print('\n=== 4.3 · QUET w THEO TIER tren val (tier CHINH THUC) ===')
PO, PC = ENS['pipe'], ENS['cnn_full']
W_VAL, W_ROB = {}, {}
for t, cols in TIER.items():
    sc = np.array([float(np.nanmean(ap_cols(Y_VA, (1-w)*PO + w*PC)[cols])) for w in WG])
    W_VAL[t] = float(WG[int(sc.argmax())])
    thang = np.zeros(len(WG))
    for ii in BOOT[:60]:
        s2 = [float(np.nanmean(ap_cols(Y_VA[ii], ((1-w)*PO + w*PC)[ii])[cols])) for w in WG]
        thang[int(np.argmax(s2))] += 1
    W_ROB[t] = float(WG[int(thang.argmax())])
    print(f'  {t:<7} argmax {W_VAL[t]:.2f} (AP {sc.max():.4f}) | ROBUST {W_ROB[t]:.2f}'
          f' (thang {int(thang.max())}/60) | da khoa 0.40' + ('/0.00' if t == 'tail' else ''))
K_CO = 40.0     # co ngot ve prior 0.40; k lon = tin prior hon
W_CO = {t: round(((len(TIER[t]) * W_ROB[t] + K_CO * 0.40) / (len(TIER[t]) + K_CO)) / 0.05) * 0.05
        for t in TIER}
print(f'  sau CO NGOT ve 0.40 (k={K_CO:.0f}): ' + str({t: round(v, 2) for t, v in W_CO.items()}))

print('\n=== 4.4 · BANG TONG HOP tren val ===')
PA = {'khoa 0.40/0.40/0.00 (tier CU)': tron(PO, PC, W_TIER, TIER_TA),
      'w quet tren val (tier moi)':    tron(PO, PC, W_VAL, TIER),
      'w ROBUST (tier moi)':           tron(PO, PC, W_ROB, TIER),
      'w CO NGOT (tier moi)':          tron(PO, PC, W_CO,  TIER),
      'duong ong thuan':               PO}
if 'cnn_ms' in ENS:
    _w3 = {t: 0.25 for t in TIER}
    PA['3 chieu (pipe+cnn+cnn_ms)'] = tron(tron(PO, ENS['cnn_full'], W_ROB, TIER),
                                           ENS['cnn_ms'], _w3, TIER)
print(f'{"phuong an":<34}{"macro":>9}{"head":>9}{"medium":>9}{"tail":>9}')
KQ = {}
for nm, P in PA.items():
    a = ap_cols(Y_VA, P); KQ[nm] = float(np.nanmean(a))
    print(f'  {nm:<32}{KQ[nm]:>9.4f}' + ''.join(f'{float(np.nanmean(a[TIER[t]])):>9.4f}'
          for t in ('head', 'medium', 'tail')))
_bt = max(KQ, key=KQ.get)
print(f'\n  TOT NHAT tren val: `{_bt}`  {KQ[_bt]:.4f}'
      f'  (hon ban khoa {KQ[_bt]-KQ["khoa 0.40/0.40/0.00 (tier CU)"]:+.4f})')
print(f'  san nhieu lap lai = 0.0007 -> chenh duoi muc do KHONG doc duoc')

=== 4.1 · TRUNG BINH HANG vs TRUNG BINH XAC SUAT ===
  pipe       xac suat 0.2663 | hang 0.2644   -0.0019
  cnn_full   xac suat 0.2463 | hang 0.2388   -0.0075
  cnn_ms     xac suat 0.2419 | hang 0.2366   -0.0053

=== 4.2 · TRONG SO THEO EPOCH (hien tai: DEU) ===
  deu     pipe 0.2663 | cnn_full 0.2463 | cnn_ms 0.2419
  cuoi5   pipe 0.2621 | cnn_full 0.2476 | cnn_ms 0.2419
  cuoi3   pipe 0.2603 | cnn_full 0.2459 | cnn_ms 0.2424
  tuyen   pipe 0.2643 | cnn_full 0.2477 | cnn_ms 0.2422

=== 4.3 · QUET w THEO TIER tren val (tier CHINH THUC) ===
  head    argmax 0.45 (AP 0.5652) | ROBUST 0.45 (thang 42/60) | da khoa 0.40
  medium  argmax 0.40 (AP 0.2342) | ROBUST 0.40 (thang 31/60) | da khoa 0.40
  tail    argmax 0.30 (AP 0.1397) | ROBUST 0.35 (thang 20/60) | da khoa 0.40/0.00
  sau CO NGOT ve 0.40 (k=40): {'head': 0.4, 'medium': 0.4, 'tail': 0.4}

=== 4.4 · BANG TONG HOP tren val ===
phuong an                             macro     head   medium     tail
  khoa 0.40/0.40/0.00 (tier CU)      

## C5 · Tổng hợp — cấu hình đề xuất cho lần đọc test kế tiếp

In [6]:
print('=' * 74); print('TONG HOP'); print('=' * 74)
print(f'\n1. TUAN THU     khong pham nhan zero-shot; nhan dev/test cung nguon voi train')
_ct = {k: len(v) for k, v in TIER.items()}
_cu = {k: len(v) for k, v in TIER_TA.items()}
print(f'2. TIER         chinh thuc {_ct} != tier cu cua ta {_cu}')
if CO_TEST:
    _g = {t: (VD[t] - float(np.nanmean(apT[TIER[t]]))) * len(TIER[t]) / C
          for t in ('head', 'medium', 'tail')}
    _d = max(_g, key=_g.get)
    print(f'3. KHOANG CACH  tier `{_d}` gong {_g[_d]/max(sum(_g.values()),1e-9):.0%} '
          f'({_g[_d]:+.4f} / {sum(_g.values()):+.4f})  -> do cong vao day')
print(f'4. NHAN         nhom A (nhan ban) {len(ban)} · nhom B (sach ma kho) {len(sach)} · '
      f'nhom C (chua co so) {len(chua)}')
print(f'5. QUET VAL     tot nhat `{_bt}` {KQ[_bt]:.4f}, '
      f'{KQ[_bt]-KQ["khoa 0.40/0.40/0.00 (tier CU)"]:+.4f} so ban khoa')

print('\n--- CAU HINH DE XUAT (chot tren val, CHUA doc test) ---')
print(f'  thanh vien : pipe (64/r2a) + cnn_full (64b/r2a)'
      + (' + cnn_ms' if '3 chieu' in _bt else ''))
print(f'  gop epoch  : deu (moi epoch x moi seed)')
print(f'  w theo tier: ' + str({t: round(W_CO[t], 2) for t in TIER}) + '  (tier CHINH THUC)')
print('\n  !! KHONG cham test trong notebook nay. Doc test MOT luot cho ca bang,')
print('     bao cao TAT CA phuong an ke ca cai thua — do la cach giu test sach.')
print('  !! Ngan sach doc test con lai: 2 lan (xem PLAN_VUOT_0281.md §7).')

json.dump(dict(tier={k: [LAB40[j] for j in v] for k, v in TIER.items()},
               w_val=W_VAL, w_robust=W_ROB, w_congot=W_CO,
               kq_val=KQ, tot_nhat=_bt,
               nhom_A=[r['lb'] for r in ban], nhom_B=[r['lb'] for r in sach],
               nhom_C=[r['lb'] for r in chua]),
          open(os.path.join(OUT_DIR, 'nb67_ket_qua.json'), 'w'), indent=1, ensure_ascii=False)
print(f"\nda luu {os.path.join(OUT_DIR, 'nb67_ket_qua.json')}")

TONG HOP

1. TUAN THU     khong pham nhan zero-shot; nhan dev/test cung nguon voi train
2. TIER         chinh thuc {'head': 9, 'medium': 15, 'tail': 16} != tier cu cua ta {'head': 13, 'medium': 16, 'tail': 11}
3. KHOANG CACH  tier `medium` gong 71% (+0.0129 / +0.0181)  -> do cong vao day
4. NHAN         nhom A (nhan ban) 6 · nhom B (sach ma kho) 5 · nhom C (chua co so) 10
5. QUET VAL     tot nhat `khoa 0.40/0.40/0.00 (tier CU)` 0.2709, +0.0000 so ban khoa

--- CAU HINH DE XUAT (chot tren val, CHUA doc test) ---
  thanh vien : pipe (64/r2a) + cnn_full (64b/r2a)
  gop epoch  : deu (moi epoch x moi seed)
  w theo tier: {'head': 0.4, 'medium': 0.4, 'tail': 0.4}  (tier CHINH THUC)

  !! KHONG cham test trong notebook nay. Doc test MOT luot cho ca bang,
     bao cao TAT CA phuong an ke ca cai thua — do la cach giu test sach.
  !! Ngan sach doc test con lai: 2 lan (xem PLAN_VUOT_0281.md §7).

da luu /content/drive/MyDrive/MIMIC_CXR/planB_out/cxrlt_2024/nb67_ket_qua.json
